# On-Sky MCMC Tutorial

This is the same idea as the global MCMC fit for the internal calibrations. I won't add much comments since the process is the same, except you use your on-sky CSV and data.

In [1]:
# IMPORTING DATA

from pyPolCal.csv_tools import read_csv_physical_model_all_bins
from importlib.resources import files
from pathlib import Path

# Defining Path to my CSVs
csvdir = Path('/home/thomasmc/pyPolCal/pyPolCal/CHARIS/datacsvs/onsky_nbs/HD293396')

# Reading in data
interleaved_values_all, interleaved_stds_all, configuration_list_all = read_csv_physical_model_all_bins(csvdir, m3=True)

/home/thomasmc/miniconda3/envs/charisenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# GENERATE MODEL
from pyPolCal.utils import generate_system_mueller_matrix

system_dict = {
    "components" : {
        "wollaston" : {
        "type" : "fitted_wollaston_function_12_28_2025",
        "properties" : {"beam": 'o'}, 
        "tag": "internal",
        },

        "nbs_rot": {
            "type": "rotator_function",
            "properties": {"pa": 90},
            "tag": "internal",
        },
        "image_rotator" : {
        "type" : "fitted_derotator_function_12_28_2025",
        "properties" : {"delta_theta":1.384e-02}, 
        "tag": "internal",
        },
        
        "hwp" : {
            "type" : "two_layer_HWP_function", # Joost 't Hart 2021 HWP model
            "properties" : {"w_SiO2":1.638, "w_MgF2":1.28, "delta_theta": -3.168e-02},
            "tag": "internal",
        },

        "altitude_rot" : {
            "type" : "rotator_function",
            "properties" : {"pa":0},
            "tag":"internal",
        },
        "M3" : {
            "type" : "SUBARU_M3_function",
            "properties" : {"delta_theta":0},
            "tag": "internal",
        },

        "parang_rot" : {
            "type" : "rotator_function",
            "properties" : {"pa":0},
            "tag":"internal",
        },
        },
}
system_mm = generate_system_mueller_matrix(system_dict)
system_mm.evaluate()

array([[ 0.48902582, -0.48857441,  0.00881718,  0.01906714],
       [ 0.48902582, -0.48857441,  0.00881718,  0.01906714],
       [ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ]])

In [11]:

# Define starting guesses

m1, b1, m2, b2 = (1.781,12.47,2.264,14.67) # from minimize
p0_dict = {
    "M3":{"m1": m1,
    "b1": b1,
    "m2": m2,
    "b2": b2
    }
}
m1_bounds = (0.5*m1, 2*m1)
m2_bounds = (0.5*m2, 2*m2)
b1_bounds = (0.5*b1, 2*b1)
b2_bounds = (0.5*b2, 2*b2)
boundslist = [m1_bounds, b1_bounds, m2_bounds, b2_bounds]

bounds_dict = {
    
    "M3" : {
        "m1": m1_bounds,
        "b1": b1_bounds,
        "m2": m2_bounds,
        "b2": b2_bounds
    }
}

# Define priors
prior_dict = {

    "M3": {
        "m1": {"type": "uniform", "kwargs": {"low":0.5*m1, "high": 2*m1}},
        "b1": {"type": "uniform", "kwargs": {"low":0.5*b1, "high": 2*b1}},
        "m2": {"type": "uniform", "kwargs": {"low":0.5*m2, "high": 2*m2}},
        "b2": {"type": "uniform", "kwargs": {"low":0.5*b2, "high": 2*b2}},
}}


In [5]:
from pyPolCal.instruments_jax import run_mcmc, process_model, process_dataset , process_errors

# Path for the h5 emcee output file
output_h5 = 'mcmc_tutorial_output_onsky.h5'

ndim = 6  # Number of parameters to fit
pool_processes = 12 # Number of CPU cores to use
nwalkers = max(2 * ndim, pool_processes * 2) # Number of walkers at least twice the number of dimensions
if nwalkers % pool_processes != 0:
    nwalkers += pool_processes - (nwalkers % pool_processes)

print(f"{nwalkers} walkers for {ndim} parameters")
sampler, p_keys = run_mcmc(p0_dict, system_mm, interleaved_values_all,configuration_list_all,prior_dict,bounds_dict,output_h5,errors=interleaved_stds_all,nwalkers=nwalkers,pool_processes=pool_processes,process_model=process_model, process_dataset=process_dataset,process_errors=process_errors,nsteps=7000,test_plot=False, include_sums=False)

24 walkers for 6 parameters
Initial log-likelihood: 3095.451697674448


  0%|          | 5/7000 [00:12<4:40:43,  2.41s/it]

emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:


emcee: Exception while calling your likelihood function:  params:  params:  params:  
 emcee: Exception while calling your likelihood function:  params:emcee: Exception while calling your likelihood function:
 
  params:emcee: Exception while calling your likelihood function:   params: 
  params:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function: 

emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:  params:  params:
 
   params:
  params:   params:  [ 1.77962448 12.47150336  2.25971523 14.66968789 -3.00169735][ 1.77832686 12.47314021  2.28645122 14.67380574 -2.98870951][ 1.77950142 12.47026272  2.26298126 14.66904661 -3.00050331][ 1.78271968 

KeyboardInterrupt: 

(the excessive outputs are from me running this notebook remotely on screen, it's not like this if you just run the notebook normally)

In [ ]:
# check results
from pyPolCal.plotting import summarize_median_posterior
output_h5 = 'mcmc_tutorial_output_onsky.h5'
summarize_median_posterior(output_h5,p0_dict,step_range=(800,1000))

Posterior Medians and 1 sigma Credible Intervals:
M3,m1: 1.06162 (+0.30646/-0.13167)
M3,b1: 9.14309 (+2.12699/-0.87805)
M3,m2: 1.56181 (+0.83782/-0.32167)
M3,b2: 12.31862 (+4.62774/-1.98567)


{'M3': {'m1': {'median': np.float64(1.061624169503271),
   '-1sigma': np.float64(0.13166503890595616),
   '+1sigma': np.float64(0.30645779013227337)},
  'b1': {'median': np.float64(9.143094207724927),
   '-1sigma': np.float64(0.8780467821050593),
   '+1sigma': np.float64(2.126985118906344)},
  'm2': {'median': np.float64(1.5618112041264758),
   '-1sigma': np.float64(0.321666332063719),
   '+1sigma': np.float64(0.8378206243043231)},
  'b2': {'median': np.float64(12.318617666512234),
   '-1sigma': np.float64(1.985665616067939),
   '+1sigma': np.float64(4.627743669417958)}}}